# 전처리 파이프라인 — STT → CSV

**역할**: raw STT `.txt` → `data/processed/` CSV 생성  
**출력 파일**:
- `utterances_all.csv` — 전체 발화 (EDA 기준)
- `utterances_by_lecture/*.csv` — 강의별 발화
- `lecture_summary.csv` — 강의별 요약 통계


## 1. Setup


In [17]:
import re
from pathlib import Path

import pandas as pd

DATA_DIR  = Path('../data/raw')
stt_files = sorted(DATA_DIR.glob('*.txt'))
print(f'STT 파일 수: {len(stt_files)}')
for f in stt_files:
    print(f'  {f.name}')


STT 파일 수: 15
  2026-02-02_kdt-backendj-21th.txt
  2026-02-03_kdt-backendj-21th.txt
  2026-02-04_kdt-backendj-21th.txt
  2026-02-05_kdt-backendj-21th.txt
  2026-02-06_kdt-backendj-21th.txt
  2026-02-09_kdt-backendj-21th.txt
  2026-02-10_kdt-backendj-21th.txt
  2026-02-11_kdt-backendj-21th.txt
  2026-02-12_kdt-backendj-21th.txt
  2026-02-13_kdt-backendj-21th.txt
  2026-02-23_kdt-backendj-21th.txt
  2026-02-24_kdt-backendj-21th.txt
  2026-02-25_kdt-backendj-21th.txt
  2026-02-26_kdt-backendj-21th.txt
  2026-02-27_kdt-backendj-21th.txt


In [18]:
# 구간 기준 — EDA 결과에 따라 조정
intro_minutes = 30
outro_minutes = 15
print(f'도입부 기준: {intro_minutes}분 / 마무리 기준: 종료 전 {outro_minutes}분')


도입부 기준: 30분 / 마무리 기준: 종료 전 15분


## 2. 전처리 헬퍼 함수


In [33]:
# STT 포맷: <HH:MM:SS> speaker_id: 발화 텍스트
_LINE_PATTERN = re.compile(
    r'<(\d{2}:\d{2}:\d{2})>\s+(\S+):\s+(.+)'
)

def ts_to_seconds(ts: str) -> int:
    """HH:MM:SS → 초"""
    h, m, s = map(int, ts.split(':'))
    return h * 3600 + m * 60 + s


def fix_am_pm_timestamps(seconds_list: list[int],
                         min_backward_jump_sec: int = 300) -> list[int]:
    """
    이전 timestamp보다 min_backward_jump_sec 이상 크게 감소하면
    오전→오후 전환으로 간주해 +12시간 보정.
    작은 역전은 STT/정렬 오류로 간주.
    """
    if not seconds_list:
        return []

    offset = 0
    fixed  = []
    prev   = seconds_list[0]

    for sec in seconds_list:
        if sec < prev and (prev - sec) >= min_backward_jump_sec:
            offset += 12 * 3600
        fixed.append(sec + offset)
        prev = sec

    return fixed


def parse_stt(path: Path) -> pd.DataFrame:
    """
    STT 파일 → DataFrame

    컬럼:
      lecture_id  – 파일명 기반 강의 ID
      date        – 강의 날짜 (YYYY-MM-DD)
      timestamp   – 원본 HH:MM:SS
      speaker_id  – 화자 ID
      text_raw    – 원문 발화
      text_clean  – 공백 정규화 발화
      sec_raw     – timestamp → 절대 초
      sec_fixed   – 오전/오후 보정 후 절대 초
      elapsed_sec – 강의 시작 기준 경과 초
      duration_sec– 다음 발화까지 시간 차이 (마지막 행 = 0)
      word_count  – 어절 수
      char_count  – 글자 수 (공백 제외)
    """
    rows = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            m = _LINE_PATTERN.match(line)
            if not m:
                continue
            ts, speaker, text = m.group(1), m.group(2), m.group(3)
            rows.append({
                'timestamp':  ts,
                'speaker_id': speaker,
                'text_raw':   text,
                'sec_raw':    ts_to_seconds(ts),
            })

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    # 오전/오후 보정
    df['sec_fixed']   = fix_am_pm_timestamps(df['sec_raw'].tolist())
    # 강의 시작 기준 경과 초
    df['elapsed_sec'] = df['sec_fixed'] - df['sec_fixed'].iloc[0]
    # 다음 발화까지 시간 차이 (마지막 행은 0)
    df['duration_sec'] = df['sec_fixed'].diff().shift(-1).fillna(0).clip(lower=0).astype(int)

    # 텍스트 최소 정제: 공백 정규화만
    df['text_clean'] = (
        df['text_raw']
        .fillna('')
        .str.strip()
        .str.replace(r'\s+', ' ', regex=True)
    )
    df['word_count'] = df['text_clean'].str.split().str.len()
    df['char_count'] = df['text_clean'].str.replace(' ', '', regex=False).str.len()

    # 강의 ID / 날짜
    df['lecture_id'] = path.stem
    df['date']       = path.stem[:10]

    return df[[
        'lecture_id', 'date', 'timestamp', 'speaker_id',
        'text_raw', 'text_clean',
        'sec_raw', 'sec_fixed', 'elapsed_sec', 'duration_sec',
        'word_count', 'char_count',
    ]]


def split_segments(df: pd.DataFrame,
                   intro_min: int = intro_minutes,
                   outro_min: int = outro_minutes):
    """intro / middle / outro 3구간 분리"""
    total_sec = df['elapsed_sec'].max()
    intro_sec = intro_min * 60
    outro_sec = total_sec - outro_min * 60

    intro  = df[df['elapsed_sec'] <= intro_sec].copy()
    outro  = df[df['elapsed_sec'] >= outro_sec].copy()
    middle = df[(df['elapsed_sec'] > intro_sec) &
                (df['elapsed_sec'] < outro_sec)].copy()
    return intro, middle, outro


def detect_keywords(df: pd.DataFrame, keywords: list[str]) -> pd.Series:
    """발화별 키워드 탐지 여부 (bool Series) — text_clean 기준"""
    pattern = '|'.join(re.escape(k) for k in keywords)
    return df['text_clean'].str.contains(pattern, na=False)


In [34]:
def merge_utterances(df: pd.DataFrame, gap_sec: int = 3) -> pd.DataFrame:
    """
    동일 화자·짧은 갭(≤gap_sec초) 연속 발화를 하나의 행으로 합침.
    text_raw/text_clean/word_count/char_count 재계산.
    """
    if df.empty:
        return df.copy()

    rows = []
    buf  = None
    for _, r in df.iterrows():
        if buf is None:
            buf = r.to_dict()
        elif (r['speaker_id'] == buf['speaker_id'] and
              r['elapsed_sec'] - buf['elapsed_sec'] <= gap_sec):
            buf['text_raw'] += ' ' + r['text_raw']
            # 병합 후 파생 컬럼 재계산
            clean = re.sub(r'\s+', ' ', buf['text_raw']).strip()
            buf['text_clean']  = clean
            buf['word_count']  = len(clean.split())
            buf['char_count']  = len(clean.replace(' ', ''))
            buf['duration_sec'] = r['duration_sec']  # 블록 마지막 행의 gap 사용
        else:
            rows.append(buf)
            buf = r.to_dict()
    if buf:
        rows.append(buf)

    return pd.DataFrame(rows).reset_index(drop=True)


def filter_noise(df: pd.DataFrame, min_words: int = 2) -> pd.DataFrame:
    """어절 수 < min_words 인 단독 발화 제거 (STT 노이즈 필터)."""
    return df[df['word_count'] >= min_words].reset_index(drop=True)


In [35]:
# 전체 강의 파일 로드 & 파싱
lectures: list[dict] = []
for path in stt_files:
    df = parse_stt(path)
    if df.empty:
        print(f'[SKIP] {path.name}')
        continue
    intro, middle, outro = split_segments(df)
    lectures.append({
        'name':   path.stem,
        'all':    df,
        'intro':  intro,
        'middle': middle,
        'outro':  outro,
    })

print(f'\n로드 완료: {len(lectures)}개 강의')
for lec in lectures:
    d = lec['all']
    print(
        f"  [{lec['name']}]  "
        f"발화={len(d)}행  "
        f"길이={d['elapsed_sec'].max()/60:.1f}분  "
        f"화자={d['speaker_id'].nunique()}명"
    )



로드 완료: 15개 강의
  [2026-02-02_kdt-backendj-21th]  발화=1484행  길이=419.4분  화자=1명
  [2026-02-03_kdt-backendj-21th]  발화=1860행  길이=519.6분  화자=2명
  [2026-02-04_kdt-backendj-21th]  발화=1771행  길이=519.9분  화자=1명
  [2026-02-05_kdt-backendj-21th]  발화=1656행  길이=518.6분  화자=3명
  [2026-02-06_kdt-backendj-21th]  발화=1671행  길이=404.7분  화자=1명
  [2026-02-09_kdt-backendj-21th]  발화=1555행  길이=420.4분  화자=1명
  [2026-02-10_kdt-backendj-21th]  발화=1672행  길이=520.4분  화자=1명
  [2026-02-11_kdt-backendj-21th]  발화=1560행  길이=520.1분  화자=1명
  [2026-02-12_kdt-backendj-21th]  발화=986행  길이=267.0분  화자=2명
  [2026-02-13_kdt-backendj-21th]  발화=1543행  길이=519.8분  화자=2명
  [2026-02-23_kdt-backendj-21th]  발화=1469행  길이=410.0분  화자=1명
  [2026-02-24_kdt-backendj-21th]  발화=1578행  길이=519.9분  화자=1명
  [2026-02-25_kdt-backendj-21th]  발화=1298행  길이=519.2분  화자=3명
  [2026-02-26_kdt-backendj-21th]  발화=1017행  길이=520.3분  화자=3명
  [2026-02-27_kdt-backendj-21th]  발화=1636행  길이=519.9분  화자=1명


## 3. 파싱 결과 미리보기


In [36]:
lectures[0]['all'].head(3)

,lecture_id,date,timestamp,speaker_id,text_raw,text_clean,sec_raw,sec_fixed,elapsed_sec,duration_sec,word_count,char_count
0,2026-02-02_kdt-backendj-21th,2026-02-02,09:11:17,b54f46b0,여러분 오늘 수업 진행하도록 하겠습니다. 저희가 이제 오늘 1차 잡바 마지막 날입니...,여러분 오늘 수업 진행하도록 하겠습니다. 저희가 이제 오늘 1차 잡바 마지막 날입니...,33077,33077,0,0,13,40
1,2026-02-02_kdt-backendj-21th,2026-02-02,09:11:17,b54f46b0,지난 시간에 제너릭 타입을 이용을 해서 커스텀 컬렉션을 활용한 크루드 방법을 학습 ...,지난 시간에 제너릭 타입을 이용을 해서 커스텀 컬렉션을 활용한 크루드 방법을 학습 ...,33077,33077,0,1,22,65
2,2026-02-02_kdt-backendj-21th,2026-02-02,09:11:18,b54f46b0,"NIO 패키지가 있고 그 다음에 NIO2라는 패키지를 가지고 있는데, 이 NI2는 ...","NIO 패키지가 있고 그 다음에 NIO2라는 패키지를 가지고 있는데, 이 NI2는 ...",33078,33078,1,15,28,83


In [38]:
for lec in lectures:
    df = lec['all']
    print(len(df[df['text_raw'] != df['text_clean']]))

0
0
0
0
0
0
0
0
0
0
0
0
0
0
0


## 4. CSV 내보내기


In [39]:
# ── CSV 내보내기 ────────────────────────────────────────────────
PROCESSED_DIR  = Path('../data/processed')
BY_LECTURE_DIR = PROCESSED_DIR / 'utterances_by_lecture'
PROCESSED_DIR.mkdir(exist_ok=True)
BY_LECTURE_DIR.mkdir(exist_ok=True)

# utterances_all.csv — EDA 기준 데이터
all_df = pd.concat([lec['all'] for lec in lectures], ignore_index=True)
all_df.to_csv(PROCESSED_DIR / 'utterances_all.csv', index=False, encoding='utf-8-sig')

# utterances_by_lecture/*.csv — 강의별 파일
for lec in lectures:
    lec['all'].to_csv(
        BY_LECTURE_DIR / f"{lec['name']}.csv",
        index=False, encoding='utf-8-sig',
    )

# lecture_summary.csv — 강의별 비교용 요약
summary_rows = []
for lec in lectures:
    d = lec['all']
    summary_rows.append({
        'lecture_id':         lec['name'],
        'date':               lec['name'][:10],
        'utterance_count':    len(d),
        'total_min':          round(d['elapsed_sec'].max() / 60, 1),
        'total_words':        int(d['word_count'].sum()),
        'avg_word_count':     round(d['word_count'].mean(), 1),
        'avg_duration_sec':   round(d['duration_sec'].mean(), 1),
        'speaker_count':      d['speaker_id'].nunique(),
    })
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(PROCESSED_DIR / 'lecture_summary.csv', index=False, encoding='utf-8-sig')

print(f'utterances_all.csv    : {len(all_df):,}행')
print(f'utterances_by_lecture : {len(lectures)}개 파일')
print(f'lecture_summary.csv   : {len(summary_df)}강의')
display(summary_df)


utterances_all.csv    : 22,756행
utterances_by_lecture : 15개 파일
lecture_summary.csv   : 15강의


,lecture_id,date,utterance_count,total_min,total_words,avg_word_count,avg_duration_sec,speaker_count
0,2026-02-02_kdt-backendj-21th,2026-02-02,1484,419.4,20449,13.8,17.0,1
1,2026-02-03_kdt-backendj-21th,2026-02-03,1860,519.6,24747,13.3,16.8,2
2,2026-02-04_kdt-backendj-21th,2026-02-04,1771,519.9,23424,13.2,17.6,1
3,2026-02-05_kdt-backendj-21th,2026-02-05,1656,518.6,21738,13.1,18.8,3
4,2026-02-06_kdt-backendj-21th,2026-02-06,1671,404.7,22095,13.2,14.5,1
5,2026-02-09_kdt-backendj-21th,2026-02-09,1555,420.4,21571,13.9,16.2,1
6,2026-02-10_kdt-backendj-21th,2026-02-10,1672,520.4,23124,13.8,18.7,1
7,2026-02-11_kdt-backendj-21th,2026-02-11,1560,520.1,20614,13.2,20.0,1
8,2026-02-12_kdt-backendj-21th,2026-02-12,986,267.0,12642,12.8,16.2,2
9,2026-02-13_kdt-backendj-21th,2026-02-13,1543,519.8,20691,13.4,20.2,2
